# Audio Anomaly Detection
Ensemble of Autoencoder · Isolation Forest · GMM over mel, delta-mel, and PSD features.

In [55]:
from __future__ import annotations

import random
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

try:
    import librosa
    import librosa.effects
    import librosa.feature
    LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
    print("Warning: librosa not installed — audio loading unavailable.")

try:
    from scipy.signal import welch as scipy_welch
    SCIPY_OK = True
except ImportError:
    SCIPY_OK = False
    print("Warning: scipy not installed — PSD features unavailable.")

## Config
Edit here — everything else reads from these objects.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_DIR   = Path("dataset")
CACHE_DIR     = Path(".cache");  CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR    = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
FEATURES_CACHE = CACHE_DIR / "mel_features.npz"
RESULTS_DB     = OUTPUT_DIR / "results.pkl"

RANDOM_STATE = 2137

# ── Audio ──────────────────────────────────────────────────────────────────────
@dataclass
class AudioConfig:
    sample_rate: int   = 16_000
    duration_sec: float = 10.0  
    n_fft: int         = 1024
    hop_length: int    = 512
    n_mels: int        = 128
    f_min: float       = 20.0
    f_max: float       = 8_000.0
    hp_cutoff_hz: float = 120.0
    top_db: float      = 80.0

# ── Window / Frame-level features ─────────────────────────────────────────────
@dataclass
class WindowConfig:
    frame_stack: int        = 5
    stride: int             = 3
    max_windows_per_file: int = 80

# ── Autoencoder ───────────────────────────────────────────────────────────────
@dataclass
class AEConfig:
    latent_dim: int      = 8           # was 16 — force much tighter compression
    hidden_dims: list    = field(default_factory=lambda: [256, 128, 64, 32])
    dropout: float       = 0.05
    batch_size: int      = 512
    epochs: int          = 80          # more epochs since network is smaller
    lr: float            = 5e-4        # slightly lower lr for stability
    weight_decay: float  = 1e-4
    patience: int        = 10
    min_delta: float     = 1e-5
    use_batch_norm: bool = True

# ── Isolation Forest ──────────────────────────────────────────────────────────
@dataclass
class IFConfig:
    n_estimators: int  = 200
    max_features: float = 0.5
    contamination: float = 0.05

# ── GMM ───────────────────────────────────────────────────────────────────────
@dataclass
class GMMConfig:
    n_components: int    = 16          # more components but diagonal covariance
    covariance_type: str = "diag"
    max_iter: int        = 300
    reg_covar: float     = 1e-3        # stronger regularisation

# ── PCA pre-processing ────────────────────────────────────────────────────────
@dataclass
class PCAConfig:
    n_components: int = 64
    whiten: bool      = True

# ── Ensemble ──────────────────────────────────────────────────────────────────
@dataclass
class EnsembleConfig:
    members: list = field(default_factory=lambda: [
        (1.0, "mel", "ae"),
        (0.9, "mel", "gmm"),
        (0.8, "mel", "if"),
    ])
    score_aggregation: str = "weighted_sum"  # RRF hurts when members disagree this much

# ── Data split ────────────────────────────────────────────────────────────────
@dataclass
class SplitConfig:
    test_size: float = 0.20
    val_size: float  = 0.15

# ── Instantiate ───────────────────────────────────────────────────────────────
AUDIO    = AudioConfig()
WINDOW   = WindowConfig()
AE_CFG   = AEConfig()
IF_CFG   = IFConfig()
GMM_CFG  = GMMConfig()
PCA_CFG  = PCAConfig()
ENS_CFG  = EnsembleConfig()
SPLIT    = SplitConfig()

FeatureType = Literal["mel", "psd", "delta"]
_RNG = np.random.default_rng(RANDOM_STATE)

print("Config loaded.")

Config loaded.


## Feature Extraction

In [57]:
def load_audio(path):
    y, _ = librosa.load(str(path), sr=AUDIO.sample_rate, mono=True, duration=AUDIO.duration_sec)
    if AUDIO.hp_cutoff_hz > 0:
        y = librosa.effects.preemphasis(y, coef=0.97)
    peak = np.max(np.abs(y))
    return y / peak if peak > 0 else y


def compute_mel(y):
    S = librosa.feature.melspectrogram(
        y=y, sr=AUDIO.sample_rate, n_fft=AUDIO.n_fft, hop_length=AUDIO.hop_length,
        n_mels=AUDIO.n_mels, fmin=AUDIO.f_min, fmax=AUDIO.f_max,
    )
    return librosa.power_to_db(S, top_db=AUDIO.top_db).astype(np.float32)


def compute_delta_mel(mel, order=1, width=9):
    return librosa.feature.delta(mel, order=order, width=width).astype(np.float32)


def compute_psd(y):
    _, Pxx = scipy_welch(y, fs=AUDIO.sample_rate, nperseg=AUDIO.n_fft)
    return (10 * np.log10(Pxx + 1e-10)).astype(np.float32).reshape(-1, 1)


def mel_to_windows(mel, frame_stack=None, stride=None, max_windows=None):
    frame_stack = frame_stack or WINDOW.frame_stack
    stride      = stride      or WINDOW.stride
    max_windows = max_windows if max_windows is not None else WINDOW.max_windows_per_file

    n_feats, frames = mel.shape
    if frames == 1:
        return mel.T.astype(np.float32)
    if frames < frame_stack:
        return np.empty((0, n_feats * frame_stack), dtype=np.float32)

    starts = np.arange(0, frames - frame_stack + 1, stride)
    if max_windows and len(starts) > max_windows:
        starts = starts[_RNG.choice(len(starts), max_windows, replace=False)]

    out = np.zeros((len(starts), n_feats * frame_stack), dtype=np.float32)
    for i, s in enumerate(starts):
        out[i] = mel[:, s:s + frame_stack].reshape(-1)
    return out


def collect_windows(X, indices, feature_type="mel", max_per_file=None):
    is_psd = feature_type == "psd"
    fs = 1 if is_psd else WINDOW.frame_stack
    st = 1 if is_psd else WINDOW.stride
    wins = [
        w for i in indices
        for w in [mel_to_windows(X[i], fs, st, max_per_file)]
        if len(w)
    ]
    if not wins:
        return np.empty((0, X[indices[0]].shape[0] * fs), dtype=np.float32)
    return np.vstack(wins)


def build_file_index(dataset_dir):
    """
    Expects layout: <dataset_dir>/<id_XX_SNRdb>/<normal|abnormal>/*.wav
    e.g. called with dataset/fan → finds id_00_-6db/normal/00000001.wav
    """
    records = []
    for wav in sorted(Path(dataset_dir).rglob("*.wav")):
        parts = wav.relative_to(dataset_dir).parts
        if len(parts) < 3:
            continue
        id_snr, condition = parts[0], parts[1]
        m = re.match(r"id_(\w+)_(.+)", id_snr)
        machine_id = m.group(1) if m else id_snr
        snr        = m.group(2) if m else "unknown"
        records.append({
            "path": str(wav),
            "machine_type": dataset_dir.name,   # "fan", "pump", etc.
            "snr": snr,
            "machine_id": machine_id,
            "label": 1 if condition.lower() in {"abnormal", "anomaly"} else 0,
        })
    return pd.DataFrame(records)


def extract_and_cache(index_df, cache_path=FEATURES_CACHE, force=False):
    if cache_path.exists() and not force:
        print(f"Loading features from cache: {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return {k: data[k] for k in data.files}

    n = len(index_df)
    X_mel = np.empty(n, dtype=object)
    X_delta = np.empty(n, dtype=object)
    X_psd = np.empty(n, dtype=object)
    y = np.zeros(n, dtype=np.int8)
    paths = np.empty(n, dtype=object)

    for idx, row in index_df.reset_index(drop=True).iterrows():
        try:
            audio = load_audio(row["path"])
            mel = compute_mel(audio)
            X_mel[idx]   = mel
            X_delta[idx] = compute_delta_mel(mel)
            X_psd[idx]   = compute_psd(audio)
        except Exception as e:
            print(f"  Skipping {row["path"]}: {e}")
            X_mel[idx]   = np.zeros((AUDIO.n_mels, 1), dtype=np.float32)
            X_delta[idx] = np.zeros((AUDIO.n_mels, 1), dtype=np.float32)
            X_psd[idx]   = np.zeros((513, 1), dtype=np.float32)
        y[idx]     = int(row["label"])
        paths[idx] = row["path"]
        if (idx + 1) % 100 == 0:
            print(f"  {idx + 1}/{n} files processed")

    payload = dict(X_mel=X_mel, X_delta=X_delta, X_psd=X_psd, y=y, paths=paths)
    np.savez_compressed(cache_path, **payload)
    print(f"Cache saved → {cache_path}")
    return payload


print("Feature extraction functions defined.")

Feature extraction functions defined.


## Data Splitting

In [58]:
def build_group_ids(df):
    """Group key = machine_type + machine_id, one group per physical unit."""
    return (df["machine_type"] + "_" + df["machine_id"]).values.astype(str)


def split_groups(n_samples, y, groups):
    dummy = np.arange(n_samples)
    train_all, test_idx = next(
        GroupShuffleSplit(1, test_size=SPLIT.test_size, random_state=RANDOM_STATE)
        .split(dummy, y, groups)
    )
    train_rel, val_rel = next(
        GroupShuffleSplit(1, test_size=SPLIT.val_size, random_state=RANDOM_STATE)
        .split(dummy[train_all], y[train_all], groups[train_all])
    )
    train_idx, val_idx = train_all[train_rel], train_all[val_rel]
    print(f"Split  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")
    print(f"       normal train={(y[train_idx]==0).sum()}  abnormal train={(y[train_idx]==1).sum()}")
    return train_idx, val_idx, test_idx


print("Splitting functions defined.")

Splitting functions defined.


## Models

In [59]:
# ── Autoencoder ───────────────────────────────────────────────────────────────

def _block(in_d, out_d, dropout, bn):
    layers = [nn.Linear(in_d, out_d)]
    if bn: layers.append(nn.BatchNorm1d(out_d))
    layers.append(nn.GELU())
    if dropout > 0: layers.append(nn.Dropout(dropout))
    return nn.Sequential(*layers)


class ResBlock(nn.Module):
    def __init__(self, dim, dropout, bn):
        super().__init__()
        self.block = _block(dim, dim, dropout, bn)
    def forward(self, x):
        return self.block(x) + x


class DeepAE(nn.Module):
    def __init__(self, input_dim, cfg=AE_CFG):
        super().__init__()
        hd = cfg.hidden_dims
        ld, dr, bn = cfg.latent_dim, cfg.dropout, cfg.use_batch_norm
        self.input_dim = input_dim
        self.latent_dim = ld

        enc = []
        prev = input_dim
        for h in hd:
            enc.append(_block(prev, h, dr, bn)); prev = h
        enc += [ResBlock(prev, dr, bn), nn.Linear(prev, ld)]
        self.encoder = nn.Sequential(*enc)

        dec = [nn.Linear(ld, hd[-1])]
        if bn: dec.append(nn.BatchNorm1d(hd[-1]))
        dec.append(nn.GELU())
        dec.append(ResBlock(hd[-1], dr, bn))
        prev = hd[-1]
        for h in reversed(hd[:-1]):
            dec.append(_block(prev, h, dr, bn)); prev = h
        dec.append(nn.Linear(hd[0], input_dim))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x): return self.decoder(self.encoder(x))


class AutoencoderDetector:
    def __init__(self, input_dim, feature_type="mel"):
        self.feature_type = feature_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.scaler = StandardScaler()
        self.model  = DeepAE(input_dim).to(self.device)

    def fit(self, X_source, train_normal_idx, val_normal_idx=None):
        torch.manual_seed(RANDOM_STATE)
        cfg = AE_CFG

        X_train = self.scaler.fit_transform(
            collect_windows(X_source, train_normal_idx, self.feature_type))
        if not len(X_train):
            raise ValueError("No training windows.")

        X_val = None
        if val_normal_idx is not None and len(val_normal_idx):
            raw = collect_windows(X_source, val_normal_idx, self.feature_type)
            if len(raw): X_val = self.scaler.transform(raw)

        loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32)),
                            cfg.batch_size, shuffle=True)
        val_loader = (DataLoader(TensorDataset(torch.tensor(X_val, dtype=torch.float32)),
                                 cfg.batch_size) if X_val is not None else None)

        opt  = torch.optim.AdamW(self.model.parameters(), cfg.lr, weight_decay=cfg.weight_decay)
        sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg.epochs, eta_min=1e-6)
        loss_fn = nn.MSELoss()
        best_val, best_state, patience = float("inf"), None, cfg.patience

        for epoch in range(1, cfg.epochs + 1):
            self.model.train()
            tl, tn = 0.0, 0
            for (xb,) in loader:
                xb = xb.to(self.device)
                opt.zero_grad()
                l = loss_fn(self.model(xb), xb)
                l.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                opt.step()
                tl += l.item() * len(xb); tn += len(xb)
            sch.step()
            tl /= max(tn, 1)

            vl = tl
            if val_loader:
                self.model.eval(); vl, vn = 0.0, 0
                with torch.no_grad():
                    for (xb,) in val_loader:
                        xb = xb.to(self.device)
                        vl += loss_fn(self.model(xb), xb).item() * len(xb); vn += len(xb)
                vl /= max(vn, 1)

            if epoch % 5 == 0 or epoch == 1:
                print(f"  AE[{self.feature_type}] {epoch:02d}/{cfg.epochs} train={tl:.5f} val={vl:.5f}")

            if vl < best_val - cfg.min_delta:
                best_val = vl
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                patience = cfg.patience
            else:
                patience -= 1
                if patience <= 0:
                    print(f"  Early stop at epoch {epoch} (best={best_val:.5f})"); break

        if best_state: self.model.load_state_dict(best_state)
        return self

    def score_files(self, X_source, y_source, indices):
        self.model.eval(); scores, labels = [], []
        for i in indices:
            wins = collect_windows(X_source, np.array([i]), self.feature_type, max_per_file=None)
            if not len(wins): continue
            wins = self.scaler.transform(wins)
            xb   = torch.tensor(wins, dtype=torch.float32, device=self.device)
            with torch.no_grad():
                err = torch.mean((self.model(xb) - xb) ** 2, dim=1).cpu().numpy()
            scores.append(float(err.mean())); labels.append(int(y_source[i]))
        return np.array(scores, dtype=np.float64), np.array(labels, dtype=np.int8)


# ── Isolation Forest ──────────────────────────────────────────────────────────

class IForestDetector:
    def __init__(self, feature_type="mel", use_pca=True):
        self.feature_type = feature_type
        self.scaler = StandardScaler()
        self.pca    = PCA(PCA_CFG.n_components, whiten=PCA_CFG.whiten,
                          random_state=RANDOM_STATE) if use_pca else None
        self.model = IsolationForest(n_estimators=IF_CFG.n_estimators, max_features=IF_CFG.max_features,
                                    contamination=IF_CFG.contamination,
                                    random_state=RANDOM_STATE, n_jobs=-1)

    def fit(self, X_source, train_normal_idx, val_normal_idx=None):
        X = self.scaler.fit_transform(collect_windows(X_source, train_normal_idx, self.feature_type))
        if self.pca is not None:
            X = self.pca.fit_transform(X)
            print(f"  IF[{self.feature_type}] PCA explained var: {100*self.pca.explained_variance_ratio_.sum():.1f}%")
        self.model.fit(X)
        print(f"  IF[{self.feature_type}] fitted on {len(X)} windows.")
        return self

    def score_files(self, X_source, y_source, indices):
        scores, labels = [], []
        for i in indices:
            wins = collect_windows(X_source, np.array([i]), self.feature_type, max_per_file=None)
            if not len(wins): continue
            wins = self.scaler.transform(wins)
            if self.pca: wins = self.pca.transform(wins)
            scores.append(float((-self.model.score_samples(wins)).mean()))
            labels.append(int(y_source[i]))
        return np.array(scores, dtype=np.float64), np.array(labels, dtype=np.int8)


# ── GMM ───────────────────────────────────────────────────────────────────────

class GMMDetector:
    def __init__(self, feature_type="mel", use_pca=True):
        self.feature_type = feature_type
        self.scaler = StandardScaler()
        self.pca    = PCA(PCA_CFG.n_components, whiten=PCA_CFG.whiten,
                          random_state=RANDOM_STATE) if use_pca else None
        self.model  = GaussianMixture(GMM_CFG.n_components, covariance_type=GMM_CFG.covariance_type,
                                      max_iter=GMM_CFG.max_iter, reg_covar=GMM_CFG.reg_covar,
                                      random_state=RANDOM_STATE)

    def fit(self, X_source, train_normal_idx, val_normal_idx=None):
        X = self.scaler.fit_transform(collect_windows(X_source, train_normal_idx, self.feature_type))
        if self.pca: X = self.pca.fit_transform(X)
        self.model.fit(X)
        print(f"  GMM[{self.feature_type}] BIC={self.model.bic(X):.1f}  AIC={self.model.aic(X):.1f}")
        return self

    def score_files(self, X_source, y_source, indices):
        scores, labels = [], []
        for i in indices:
            wins = collect_windows(X_source, np.array([i]), self.feature_type, max_per_file=None)
            if not len(wins): continue
            wins = self.scaler.transform(wins)
            if self.pca: wins = self.pca.transform(wins)
            scores.append(float((-self.model.score_samples(wins)).mean()))
            labels.append(int(y_source[i]))
        return np.array(scores, dtype=np.float64), np.array(labels, dtype=np.int8)


print("Models defined.")

Models defined.


## Ensemble

In [60]:
def safe_auc(labels, scores):
    if len(np.unique(labels)) < 2: return float("nan")
    return float(roc_auc_score(labels, scores))


def compute_threshold(labels, scores, fallback_q=0.99):
    if not len(scores): return float("nan")
    if len(np.unique(labels)) < 2: return float(np.quantile(scores, fallback_q))
    fpr, tpr, thresholds = roc_curve(labels, scores)
    return float(thresholds[np.argmax(tpr - fpr)])


def reciprocal_rank_fusion(score_matrix, weights, k=60):
    rrf = np.zeros(score_matrix.shape[0])
    for d in range(score_matrix.shape[1]):
        ranks = np.argsort(np.argsort(-score_matrix[:, d])) + 1
        rrf  += weights[d] / (k + ranks)
    return rrf


class AnomalyEnsemble:
    def __init__(self):
        self.members = []   # list of (detector, weight, name)

    def add(self, detector, weight, name):
        self.members.append((detector, weight, name))
        print(f"  + {name}  weight={weight}")

    def fit(self, X_mel, X_delta, X_psd, y, train_idx, val_idx):
        X_map = dict(mel=X_mel, delta=X_delta, psd=X_psd)
        train_normal = train_idx[y[train_idx] == 0]
        val_normal   = val_idx[y[val_idx]   == 0]
        for det, w, name in self.members:
            print(f"Fitting {name} …")
            X = X_map[det.feature_type]
            det.fit(X, train_normal, val_normal)
            scores, labels = det.score_files(X, y, val_idx)
            auc = safe_auc(labels, scores)
            print(f"  └─ {name}  val AUC={auc:.4f}")
        return self

    def predict(self, X_mel, X_delta, X_psd, y, indices):
        X_map = dict(mel=X_mel, delta=X_delta, psd=X_psd)
        all_scores, ref_labels = [], None
        for det, w, name in self.members:
            s, l = det.score_files(X_map[det.feature_type], y, indices)
            all_scores.append(s)
            if ref_labels is None: ref_labels = l

        score_matrix = np.column_stack(all_scores)
        normed = np.zeros_like(score_matrix)
        for d in range(score_matrix.shape[1]):
            col = score_matrix[:, d]
            normed[:, d] = (col - col.min()) / (col.max() - col.min() + 1e-12)

        weights = np.array([w for _, w, _ in self.members])
        if ENS_CFG.score_aggregation == "rank_fusion":
            fused = reciprocal_rank_fusion(normed, weights)
        else:
            fused = (normed * weights).sum(axis=1)

        return fused, ref_labels


def build_ensemble():
    ens = AnomalyEnsemble()
    dim_mel   = AUDIO.n_mels * WINDOW.frame_stack
    dim_psd   = AUDIO.n_fft // 2 + 1
    dim_delta = dim_mel
    dim_map   = dict(mel=dim_mel, psd=dim_psd, delta=dim_delta)

    factory = {
        "ae":  lambda ft: AutoencoderDetector(dim_map[ft], feature_type=ft),
        "if":  lambda ft: IForestDetector(feature_type=ft, use_pca=(ft == "mel")),
        "gmm": lambda ft: GMMDetector(feature_type=ft, use_pca=(ft == "mel")),
    }
    for weight, feat_type, model_type in ENS_CFG.members:
        det = factory[model_type](feat_type)
        ens.add(det, weight, f"{model_type.upper()}[{feat_type}]")
    return ens


print("Ensemble utilities defined.")

Ensemble utilities defined.


## Run — Load Data → Train → Evaluate

In [61]:
# ── Seed ─────────────────────────────────────────────────────────────────────
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# ── Load & cache features for the chosen machine ──────────────────────────────
MACHINE = "fan"
index_df = build_file_index(DATASET_DIR / MACHINE)
print(f"Files for {MACHINE}: {len(index_df)}  "
      f"(normal={(index_df.label==0).sum()}  abnormal={(index_df.label==1).sum()})")
print(index_df[["machine_id","snr","label"]].value_counts().sort_index().to_string())

feats = extract_and_cache(index_df)
X_mel, X_delta, X_psd = feats["X_mel"], feats["X_delta"], feats["X_psd"]
y = feats["y"]

Files for fan: 8343  (normal=6102  abnormal=2241)
machine_id  snr   label
00          -6db  0        1011
                  1         407
            0db   0        1011
                  1         407
02          -6db  0        1016
                  1         359
            0db   0        1016
                  1         359
04          0db   0        1033
                  1         348
06          0db   0        1015
                  1         361
  100/8343 files processed
  200/8343 files processed
  300/8343 files processed
  400/8343 files processed
  500/8343 files processed
  600/8343 files processed
  700/8343 files processed
  800/8343 files processed
  900/8343 files processed
  1000/8343 files processed
  1100/8343 files processed
  1200/8343 files processed
  1300/8343 files processed
  1400/8343 files processed
  1500/8343 files processed
  1600/8343 files processed
  1700/8343 files processed
  1800/8343 files processed
  1900/8343 files processed
  2000/8343 files p

In [ ]:
# ── Seed ─────────────────────────────────────────────────────────────────────
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# ── Load & cache features for the chosen machine ──────────────────────────────
MACHINE = "fan"
index_df = build_file_index(DATASET_DIR / MACHINE)
print(f"Files for {MACHINE}: {len(index_df)}  "
      f"(normal={(index_df.label==0).sum()}  abnormal={(index_df.label==1).sum()})")
print(index_df[["machine_id","snr","label"]].value_counts().sort_index().to_string())

feats = extract_and_cache(index_df)
X_mel, X_delta, X_psd = feats["X_mel"], feats["X_delta"], feats["X_psd"]
y = feats["y"]

Files for fan: 8343  (normal=6102  abnormal=2241)
machine_id  snr   label
00          -6db  0        1011
                  1         407
            0db   0        1011
                  1         407
02          -6db  0        1016
                  1         359
            0db   0        1016
                  1         359
04          0db   0        1033
                  1         348
06          0db   0        1015
                  1         361
  100/8343 files processed
  200/8343 files processed
  300/8343 files processed
  400/8343 files processed
  500/8343 files processed
  600/8343 files processed
  700/8343 files processed
  800/8343 files processed
  900/8343 files processed
  1000/8343 files processed
  1100/8343 files processed
  1200/8343 files processed
  1300/8343 files processed
  1400/8343 files processed
  1500/8343 files processed
  1600/8343 files processed
  1700/8343 files processed
  1800/8343 files processed
  1900/8343 files processed
  2000/8343 files p

## Per-ID training loop

In [62]:
import itertools

RUN_IDS  = sorted(index_df.machine_id.unique())
RUN_SNRS = sorted(index_df.snr.unique())


all_results = []

for machine_id, snr in itertools.product(RUN_IDS, RUN_SNRS):

    mask = (index_df.machine_id == machine_id) & (index_df.snr == snr)
    sub  = index_df[mask].reset_index(drop=True)

    if len(sub) == 0:
        print(f"  Skipping {machine_id}/{snr} — no files found")
        continue

    n_normal   = (sub.label == 0).sum()
    n_abnormal = (sub.label == 1).sum()
    print(f"\n{'='*50}")
    print(f"ID={machine_id}  SNR={snr}  normal={n_normal}  abnormal={n_abnormal}")

    if n_normal < 10 or n_abnormal < 2:
        print("  Skipping — not enough files")
        continue

    sub_idx   = sub.index.values
    sub_y     = y[sub_idx]
    sub_mel   = X_mel[sub_idx]
    sub_delta = X_delta[sub_idx]
    sub_psd   = X_psd[sub_idx]

    normal_idx   = np.where(sub_y == 0)[0]
    abnormal_idx = np.where(sub_y == 1)[0]

    rng = np.random.default_rng(RANDOM_STATE)
    rng.shuffle(normal_idx)

    n_val  = max(1, int(len(normal_idx) * 0.20))   # was 0.15
    n_test = max(1, int(len(normal_idx) * 0.20))   # was 0.20 — keep same

    val_normal_idx   = normal_idx[:n_val]
    test_normal_idx  = normal_idx[n_val:n_val + n_test]
    train_normal_idx = normal_idx[n_val + n_test:]

    # Use only a balanced subset of abnormals for val threshold tuning
    # All abnormals still go to test
    n_abn_val = min(len(abnormal_idx), n_val)
    abn_val_idx  = abnormal_idx[:n_abn_val]
    abn_test_idx = abnormal_idx                     # all abnormals in test

    val_idx_local  = np.concatenate([val_normal_idx, abn_val_idx])
    test_idx_local = np.concatenate([test_normal_idx, abn_test_idx])

    sub_y_val  = sub_y[val_idx_local]
    sub_y_test = sub_y[test_idx_local]

    print(f"  train normal={len(train_normal_idx)}  val={len(val_idx_local)}  test={len(test_idx_local)}")

    ens = build_ensemble()
    for det, w, name in ens.members:
        X_src = dict(mel=sub_mel, delta=sub_delta, psd=sub_psd)[det.feature_type]
        det.fit(X_src, train_normal_idx, val_normal_idx)

    for det, w, name in ens.members:
        X_src = dict(mel=sub_mel, delta=sub_delta, psd=sub_psd)[det.feature_type]
        s, l = det.score_files(X_src, sub_y, val_idx_local)
        print(f"    {name} individual val AUC = {safe_auc(l, s):.4f}")

    val_scores,  val_labels  = ens.predict(sub_mel, sub_delta, sub_psd, sub_y, val_idx_local)
    test_scores, test_labels = ens.predict(sub_mel, sub_delta, sub_psd, sub_y, test_idx_local)

    val_auc   = safe_auc(val_labels,  val_scores)
    test_auc  = safe_auc(test_labels, test_scores)
    threshold = compute_threshold(val_labels, val_scores)
    test_pred = (test_scores >= threshold).astype(int)

    print(f"  Val AUC={val_auc:.4f}  Test AUC={test_auc:.4f}  Threshold={threshold:.5f}")
    print(classification_report(test_labels, test_pred,
                                 target_names=["normal","abnormal"], zero_division=0))

    all_results.append(dict(
        machine=MACHINE, machine_id=machine_id, snr=snr,
        val_auc=val_auc, test_auc=test_auc,
        n_train_normal=len(train_normal_idx),
        n_test=len(test_idx_local),
    ))


ID=00  SNR=-6db  normal=1011  abnormal=407
  train normal=607  val=404  test=609
  + AE[mel]  weight=1.0
  + GMM[mel]  weight=0.9
  + IF[mel]  weight=0.8
  AE[mel] 01/80 train=0.51873 val=0.34796
  AE[mel] 05/80 train=0.31164 val=0.29536
  AE[mel] 10/80 train=0.28662 val=0.27900
  AE[mel] 15/80 train=0.27777 val=0.27314
  AE[mel] 20/80 train=0.27161 val=0.26897
  AE[mel] 25/80 train=0.26770 val=0.26550
  AE[mel] 30/80 train=0.26489 val=0.26307
  AE[mel] 35/80 train=0.26278 val=0.26200
  AE[mel] 40/80 train=0.26115 val=0.26046
  AE[mel] 45/80 train=0.25999 val=0.26051
  AE[mel] 50/80 train=0.25875 val=0.26050
  AE[mel] 55/80 train=0.25824 val=0.25954
  AE[mel] 60/80 train=0.25743 val=0.25914
  AE[mel] 65/80 train=0.25728 val=0.25875
  AE[mel] 70/80 train=0.25741 val=0.25882
  AE[mel] 75/80 train=0.25706 val=0.25867
  AE[mel] 80/80 train=0.25702 val=0.25967
  GMM[mel] BIC=8025281.0  AIC=8007146.1
  IF[mel] PCA explained var: 82.8%
  IF[mel] fitted on 48560 windows.
    AE[mel] individua

## Results

In [63]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))
print(f"\nMean test AUC : {results_df.test_auc.mean():.4f}")
print(f"Per-SNR mean  :\n{results_df.groupby('snr').test_auc.mean().to_string()}")

machine machine_id  snr  val_auc  test_auc  n_train_normal  n_test
    fan         00 -6db 0.591658  0.550928             607     609
    fan         00  0db 0.589673  0.550758             607     609
    fan         02 -6db 0.593385  0.577205             582     600
    fan         02  0db 0.597251  0.576683             582     600
    fan         04  0db 0.605697  0.568074             586     601
    fan         06  0db 0.594915  0.574659             583     600

Mean test AUC : 0.5664
Per-SNR mean  :
snr
-6db    0.564066
0db     0.567543


## Save Results

In [64]:
import json

meta_path = OUTPUT_DIR / f"{MACHINE}_meta.json"
meta_path.write_text(json.dumps(all_results, indent=2, default=float))
print(f"Results saved → {meta_path}")
print(pd.DataFrame(all_results).to_string(index=False))

Results saved → outputs\fan_meta.json
machine machine_id  snr  val_auc  test_auc  n_train_normal  n_test
    fan         00 -6db 0.591658  0.550928             607     609
    fan         00  0db 0.589673  0.550758             607     609
    fan         02 -6db 0.593385  0.577205             582     600
    fan         02  0db 0.597251  0.576683             582     600
    fan         04  0db 0.605697  0.568074             586     601
    fan         06  0db 0.594915  0.574659             583     600
